Parallel Merge Sort:

Concept:

- Partition a large unsorted list into chunks.
- Use tasks to sort each partition concurrently.
- Organize a tree of actors to progressively merge sorted partitions.

Internals to Explore:

- Task dependency management where merge tasks depend on sorted partitions.
- The efficiency of data transfers via Ray's distributed object store during merging.
- The performance of actor-to-actor communication in the merge tree hierarchy.

Opportunities for Improvement:

- Optimize how data is chunked and passed between actors to reduce serialization overhead.
- Evaluate scheduling policies for merging tasks to improve throughput.

In [1]:
# Use a remote task to sort a partition.
import ray
import random

ray.init()

@ray.remote
def sort_partition(data):
    return sorted(data)

2025-03-22 21:39:20,766	INFO util.py:154 -- Outdated packages:
  ipywidgets==7.8.1 found, needs ipywidgets>=8
Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2025-03-22 21:39:21,544	INFO worker.py:1843 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


In [2]:
# Define a merge actor that will merge two sorted lists. 
# An actor here demonstrates persistent state and actor-specific scheduling.
@ray.remote
class MergeActor:
    def merge(self, left, right):
        i = 0
        j = 0
        result = []
        while i < len(left) and j < len(right):
            if left[i] < right[j]:
                result.append(left[i])
                i += 1
            else:
                result.append(right[j])
                j += 1
        result.extend(left[i:])
        result.extend(right[j:])
        return result

In [3]:
# Orchestrating the Merge Sort
# The main function partitions the unsorted list, sorts each partition in parallel, 
# and then repeatedly merges the sorted partitions until only one sorted list remains.
def parallel_merge_sort(data, num_partitions):
    # Partition the data
    chunk_size = len(data) // num_partitions
    partitions = [data[i * chunk_size: (i + 1) * chunk_size] for i in range(num_partitions)]
    
    # If there is any remainder, add it to the last partition.
    if len(data) % num_partitions != 0:
        partitions[-1].extend(data[num_partitions * chunk_size:])
    
    # Sort partitions concurrently using remote tasks.
    sorted_refs = [sort_partition.remote(part) for part in partitions]
    
    # Retrieve sorted partitions.
    sorted_partitions = ray.get(sorted_refs)
    
    # Merge sorted partitions using actors in a tree-like fashion.
    while len(sorted_partitions) > 1:
        new_partitions = []
        # Merge pairs of sorted partitions.
        for i in range(0, len(sorted_partitions), 2):
            if i + 1 < len(sorted_partitions):
                merger = MergeActor.remote()
                merged_ref = merger.merge.remote(sorted_partitions[i], sorted_partitions[i+1])
                new_partitions.append(merged_ref)
            else:
                # If an odd partition remains, carry it over.
                new_partitions.append(sorted_partitions[i])
        # Get the results of this merge round.
        sorted_partitions = ray.get(new_partitions)
    
    # Return the fully sorted list.
    return sorted_partitions[0]

if __name__ == "__main__":
    # Generate a large list of random numbers.
    data = [random.randint(0, 1000) for _ in range(100)]
    num_partitions = 4  # Adjust this depending on your machine's cores.
    
    sorted_data = parallel_merge_sort(data, num_partitions)
    print("Sorted Data:", sorted_data)


Sorted Data: [0, 12, 20, 29, 33, 33, 75, 78, 91, 95, 104, 106, 151, 159, 162, 169, 170, 175, 188, 203, 211, 219, 230, 233, 250, 256, 281, 283, 287, 288, 296, 298, 310, 315, 331, 332, 335, 360, 362, 398, 408, 414, 424, 427, 441, 447, 450, 451, 452, 486, 499, 499, 514, 522, 554, 554, 578, 578, 597, 618, 641, 654, 660, 688, 690, 698, 707, 716, 722, 738, 745, 749, 751, 768, 788, 796, 804, 819, 823, 823, 825, 830, 830, 833, 835, 854, 882, 888, 888, 889, 889, 898, 903, 906, 917, 921, 929, 958, 983, 995]


In [4]:
ray.shutdown()

## Some Ideas For Improvement

- Monitor data movement
- Monitor serialization overhead